In [0]:
# Read the five raw Bronze tables

villages_df = spark.table("workspace.agri_intelligence.bronze_villages")
farmers_df = spark.table("workspace.agri_intelligence.bronze_farmers")
products_df = spark.table("workspace.agri_intelligence.bronze_products")
bills_df = spark.table("workspace.agri_intelligence.bronze_bills")
bill_items_df = spark.table("workspace.agri_intelligence.bronze_bill_items")

print("All Bronze tables loaded successfully.")

All Bronze tables loaded successfully.


In [0]:
# Clean village names and maintain a lowercase matching key

silver_villages_df = (
    villages_df
    .withColumn("village_id", trim(col("village_id")))
    .withColumn("village_name", trim(col("village_name")))
    .withColumn("normalised_village_name", lower(trim(col("village_name"))))
)


In [0]:
# Clean farmer details while keeping mobile numbers as strings

silver_farmers_df = (
    farmers_df
    .withColumn("farmer_id", trim(col("farmer_id")))
    .withColumn("farmer_name", trim(col("farmer_name")))
    .withColumn("mobile_number", trim(col("mobile_number")))
    .withColumn("village_id", trim(col("village_id")))
    .withColumn("village_name", trim(col("village_name")))
)

In [0]:
# Clean product descriptions and convert numeric product attributes

silver_products_df = (
    products_df
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("company_name", trim(col("company_name")))
    .withColumn("brand_name", trim(col("brand_name")))
    .withColumn("variety_name", trim(col("variety_name")))
    .withColumn("pack_size_value", col("pack_size_value").cast("decimal(10,2)"))
    .withColumn("selling_price_inr", col("selling_price_inr").cast("decimal(10,2)"))
)

In [0]:
# Convert bill dates and financial values into proper data types

silver_bills_df = (
    bills_df
    .withColumn("bill_id", trim(col("bill_id")))
    .withColumn("farmer_id", when(trim(col("farmer_id")) == "", None)
                .otherwise(trim(col("farmer_id"))))
    .withColumn("bill_date", to_date(col("bill_date"), "yyyy-MM-dd"))
    .withColumn("item_line_count", col("item_line_count").cast("integer"))
    .withColumn("bill_total_inr", col("bill_total_inr").cast("decimal(12,2)"))
)

In [0]:
# Convert quantities and selling amounts into numeric values

silver_bill_items_df = (
    bill_items_df
    .withColumn("bill_item_id", trim(col("bill_item_id")))
    .withColumn("bill_id", trim(col("bill_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("quantity_packs", col("quantity_packs").cast("integer"))
    .withColumn("unit_rate_inr", col("unit_rate_inr").cast("decimal(12,2)"))
    .withColumn("line_total_inr", col("line_total_inr").cast("decimal(12,2)"))
)

In [0]:
# Save cleaned datasets as Silver Delta tables

silver_villages_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.silver_villages"
)

silver_farmers_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.silver_farmers"
)

silver_products_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.silver_products"
)

silver_bills_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.silver_bills"
)

silver_bill_items_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.silver_bill_items"
)

print("All five Silver tables created successfully!")

All five Silver tables created successfully!


In [0]:
# Check Silver table row counts

print("Villages:", silver_villages_df.count())
print("Farmers:", silver_farmers_df.count())
print("Products:", silver_products_df.count())
print("Bills:", silver_bills_df.count())
print("Bill Items:", silver_bill_items_df.count())

Villages: 7
Farmers: 100
Products: 215
Bills: 420
Bill Items: 729
